# Stage 4 Missingness-Mechanism Addendum — Locked Analysis

**Scientific status:** targeted post hoc sensitivity analysis motivated by the locked Stage 4 finding. Stage 3B remains the prespecified confirmatory primary analysis. Stages 3C, Stage 4, and this addendum remain exploratory.

This notebook uses the exact saved Stage 3B outer assignments and evaluates:

1. a **missingness-only ridge** benchmark for all three outcomes;
2. broad ridge without explicit numeric/binary missingness indicators for all outcomes;
3. XGBoost without explicit missingness indicators for `NoHandJointInvolvement` only;
4. broad ridge restricted, within each outer-training split, to predictors with ≤5% missingness;
5. descriptive outcome–missingness associations and common missingness patterns.

`OnlyLargeJoints` is excluded. No resampling, class weighting, threshold optimization, new algorithm family, or outcome-guided predictor search is permitted. After this addendum, internal model development stops.

In [ ]:
# 1. Imports, versions, and deterministic settings
from __future__ import annotations
import hashlib, json, os, pickle, platform, subprocess, sys, time, warnings
from pathlib import Path
from packaging.version import Version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import yaml
from IPython.display import display, Markdown
from scipy.special import logit
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from threadpoolctl import threadpool_limits
from xgboost import XGBClassifier

if Version(sklearn.__version__) < Version('1.8') or Version(sklearn.__version__) >= Version('1.10'):
    raise RuntimeError(f'scikit-learn >=1.8,<1.10 required; found {sklearn.__version__}')
MASTER_SEED=20261020
np.random.seed(MASTER_SEED)
pd.set_option('display.max_columns',180); pd.set_option('display.max_rows',200)
pd.set_option('display.width',220); pd.set_option('display.float_format',lambda x:f'{x:,.5f}')
mpl.rcParams.update({'figure.dpi':130,'savefig.dpi':400,'font.size':10,'axes.titlesize':12,
                     'legend.fontsize':9,'figure.titlesize':14,'axes.spines.top':False,
                     'axes.spines.right':False,'svg.fonttype':'none'})
sns.set_theme(style='whitegrid',context='paper',palette='colorblind')
print('Python:',sys.version.split()[0]); print('Platform:',platform.platform())
print('pandas:',pd.__version__,'| scikit-learn:',sklearn.__version__)

In [ ]:
# 2. Project discovery, execution mode, and isolated outputs
NOTEBOOK_NAME='04M_locked_missingness_addendum.ipynb'
ENV_PROJECT = Path(os.environ['RA_PROJECT_DIR']).expanduser() if os.environ.get('RA_PROJECT_DIR') else Path.cwd(); COLAB_PROJECT=Path.cwd()
env_root=os.environ.get('RA_PROJECT_DIR')
candidates=([Path(env_root)] if env_root else [])+[ENV_PROJECT,COLAB_PROJECT,Path.cwd(),Path.cwd()/'ra_analysis',
    Path.cwd()/'stage2_outputs',Path.cwd()]
PROJECT_DIR=next((p for p in candidates if (p/'derived_data').exists() and (p/'modeling_results').exists()),None)
if PROJECT_DIR is None:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        PROJECT_DIR=next((p for p in candidates if (p/'derived_data').exists() and (p/'modeling_results').exists()),None)
    except (ImportError,ModuleNotFoundError): pass
if PROJECT_DIR is None: raise FileNotFoundError('Set RA_PROJECT_DIR to the directory containing derived_data and modeling_results.')
DERIVED_DIR=PROJECT_DIR/'derived_data'; STAGE3B_TABLES=PROJECT_DIR/'modeling_results'/'tables'
STAGE3C_DIR=PROJECT_DIR/'modeling_results_stage3c'
STAGE3C_TABLES=next((p for p in [STAGE3C_DIR/'tables',STAGE3C_DIR/'tables-3c'] if p.exists()),None)
if STAGE3C_TABLES is None: raise FileNotFoundError('Stage 3C tables or tables-3c not found')
protocol_candidates=[PROJECT_DIR/'stage4_missingness_addendum_protocol.yaml',PROJECT_DIR/'config'/'stage4_missingness_addendum_protocol.yaml',
    PROJECT_DIR.parent/'stage4_missingness_addendum_protocol.yaml',Path.cwd()/'stage4_missingness_addendum_protocol.yaml',
    ENV_PROJECT/'stage4_missingness_addendum_protocol.yaml']
PROTOCOL_PATH=next((p for p in protocol_candidates if p.exists()),None)
if PROTOCOL_PATH is None: raise FileNotFoundError('stage4_missingness_addendum_protocol.yaml not found')
with open(PROTOCOL_PATH,encoding='utf-8') as f: PROTOCOL=yaml.safe_load(f)
OUT_DIR=PROJECT_DIR/'modeling_results_stage4_missingness'
TABLE_DIR=OUT_DIR/'tables'; FIGURE_DIR=OUT_DIR/'figures'; CHECKPOINT_DIR=OUT_DIR/'checkpoints'; REPORT_DIR=OUT_DIR/'report'
for d in [OUT_DIR,TABLE_DIR,FIGURE_DIR,CHECKPOINT_DIR,REPORT_DIR]: d.mkdir(parents=True,exist_ok=True)
QUICK_MODE=os.environ.get('RA_STAGE4M_QUICK_TEST','0')=='1'
FORCE_RECOMPUTE=os.environ.get('RA_STAGE4M_FORCE_RECOMPUTE','0')=='1'
XGB_DEVICE=os.environ.get('RA_STAGE4M_XGB_DEVICE','cpu')
ACTIVE_REPEATS=[0] if QUICK_MODE else list(range(10)); OUTER_FOLDS=3 if QUICK_MODE else 5
INNER_SPLITS=2 if QUICK_MODE else 5; BOOTSTRAPS=100 if QUICK_MODE else 2000
XGB_SEARCH_ITER=4 if QUICK_MODE else 50; N_JOBS=-1; MISSINGNESS_THRESHOLD=0.05
OUTCOMES=['NegRFandACCP','NoHandJointInvolvement','AsymmetricArthritis']
print('Project:',PROJECT_DIR.resolve());print('Output:',OUT_DIR.resolve())
print('Mode:','QUICK SOFTWARE TEST' if QUICK_MODE else 'FULL LOCKED MISSINGNESS ADDENDUM')

In [ ]:
# 3. Immutable input checks and signed run definition
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

def find_first(paths,label):
    p=next((Path(x) for x in paths if Path(x).exists()),None)
    if p is None: raise FileNotFoundError(label)
    return p

required={
 'splits':find_first([STAGE3B_TABLES/'saved_outer_split_assignments.csv',STAGE3B_TABLES/'stage3b__saved_outer_split_assignments.csv'],'saved splits'),
 'stage3b_audit':find_first([STAGE3B_TABLES/'stage3b_validation_audit.csv',STAGE3B_TABLES/'stage3b__validation_audit.csv'],'Stage 3B audit'),
 'stage3c_audit':find_first([STAGE3C_TABLES/'stage3c_validation_audit.csv'],'Stage 3C audit'),
 'stage3c_repeated':find_first([STAGE3C_TABLES/'all_repeated_oof_predictions.csv'],'Stage 3C repeated predictions')}
for p in [required['stage3b_audit'],required['stage3c_audit']]:
    a=pd.read_csv(p)
    if a.empty or not a.status.eq('PASS').all(): raise AssertionError(f'Prior audit failed: {p}')
saved_splits=pd.read_csv(required['splits']).query('repeat in @ACTIVE_REPEATS and fold < @OUTER_FOLDS').copy()
original_repeated=pd.read_csv(required['stage3c_repeated']).query('repeat in @ACTIVE_REPEATS and fold < @OUTER_FOLDS').copy()
input_paths=dict(required)
for o in OUTCOMES:
    input_paths[f'{o}_data']=DERIVED_DIR/f'{o}__primary_eligible.csv'
    input_paths[f'{o}_manifest']=DERIVED_DIR/f'{o}__predictor_manifest.csv'
    for p in [input_paths[f'{o}_data'],input_paths[f'{o}_manifest']]:
        if not p.exists(): raise FileNotFoundError(p)
signature_payload={'pipeline_version':'1.0-locked-missingness-addendum','mode':'quick' if QUICK_MODE else 'full',
 'protocol_sha256':sha256_file(PROTOCOL_PATH),'inputs':{k:sha256_file(v) for k,v in input_paths.items()},
 'active_repeats':ACTIVE_REPEATS,'outer_folds':OUTER_FOLDS,'inner_splits':INNER_SPLITS,
 'bootstraps':BOOTSTRAPS,'xgb_search_iter':XGB_SEARCH_ITER,'missingness_threshold':MISSINGNESS_THRESHOLD,'seed':MASTER_SEED}
RUN_SIGNATURE=hashlib.sha256(json.dumps(signature_payload,sort_keys=True).encode()).hexdigest()
with open(OUT_DIR/'run_signature.json','w',encoding='utf-8') as f: json.dump({'run_signature':RUN_SIGNATURE,**signature_payload},f,indent=2)
print('Run signature:',RUN_SIGNATURE)

## Locked data preparation and model definitions

In [ ]:
# 4. Frozen preprocessing helpers
CATEGORY_LEVELS={'Gender':['Female','Male'],'RAOnset':['Insidious','Acute','Unknown'],'PhysicalActivity':['Moderate','Low','High']}
class MissingnessFlags(BaseEstimator,TransformerMixin):
    def fit(self,X,y=None): self.n_features_in_=X.shape[1]; return self
    def transform(self,X): return pd.isna(X).astype(float).to_numpy() if hasattr(X,'to_numpy') else np.isnan(X).astype(float)
    def get_feature_names_out(self,input_features=None):
        if input_features is None: input_features=[f'x{i}' for i in range(self.n_features_in_)]
        return np.asarray([f'{x}__missing' for x in input_features],dtype=object)

def load_outcome(outcome):
    data=pd.read_csv(DERIVED_DIR/f'{outcome}__primary_eligible.csv')
    manifest=pd.read_csv(DERIVED_DIR/f'{outcome}__predictor_manifest.csv')
    raw=manifest.loc[manifest.role.eq('primary_predictor'),'column'].tolist()
    data['RADurationDays_log1p']=np.log1p(data['RADurationDays'])
    predictors=['RADurationDays_log1p' if x=='RADurationDays' else x for x in raw]
    semantic=dict(zip(manifest.column,manifest.semantic_type)); semantic['RADurationDays_log1p']='numeric';semantic.pop('RADurationDays',None)
    if outcome in predictors or 'VisitID' in predictors or 'OnlyLargeJoints' in predictors: raise AssertionError('Leakage/excluded outcome in predictors')
    return data,predictors,semantic,raw

def make_preprocessor(data,predictors,semantic,explicit_flags):
    numeric=[x for x in predictors if semantic[x]=='numeric'];binary=[x for x in predictors if semantic[x]=='binary'];categorical=[x for x in predictors if semantic[x]=='categorical'];tr=[]
    if numeric: tr.append(('numeric',Pipeline([('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),('scaler',StandardScaler())]),numeric))
    if binary: tr.append(('binary',Pipeline([('imputer',SimpleImputer(strategy='most_frequent',keep_empty_features=True))]),binary))
    if categorical:
        categories=[]
        for col in categorical:
            if col not in CATEGORY_LEVELS: raise KeyError(f'No frozen categories for {col}')
            categories.append(CATEGORY_LEVELS[col])
        tr.append(('categorical',Pipeline([('imputer',SimpleImputer(strategy='constant',fill_value='Unknown',keep_empty_features=True)),
            ('onehot',OneHotEncoder(categories=categories,drop='first',handle_unknown='ignore',sparse_output=True))]),categorical))
    indicators=[x for x in numeric+binary if data[x].isna().any()]
    if explicit_flags and indicators: tr.append(('missing',MissingnessFlags(),indicators))
    return ColumnTransformer(tr,remainder='drop',verbose_feature_names_out=True)

def missingness_matrix(data,predictors):
    return data[predictors].isna().astype(float).rename(columns=lambda x:f'{x}__missing')

In [ ]:
# 5. Frozen strategies, search spaces, exact split plan, and checkpoints
STRATEGIES=[('missingness_only','refined_broad_ridge'),('no_explicit_flags','refined_broad_ridge'),('near_complete','refined_broad_ridge'),
            ('no_explicit_flags','xgboost')]
ridge_C=[0.01,0.1,0.5] if QUICK_MODE else np.logspace(np.log10(0.003),np.log10(0.5),21).tolist()
xgb_params={'model__max_depth':[1,2] if QUICK_MODE else [1,2,3],
 'model__learning_rate':[0.05,0.1] if QUICK_MODE else [0.01,0.03,0.05,0.1],
 'model__min_child_weight':[5] if QUICK_MODE else [5,10,20], 'model__subsample':[0.8] if QUICK_MODE else [0.7,0.85,1.0],
 'model__colsample_bytree':[0.75] if QUICK_MODE else [0.5,0.75,1.0], 'model__reg_lambda':[10] if QUICK_MODE else [1,10,50],
 'model__reg_alpha':[0] if QUICK_MODE else [0,0.1,1], 'model__n_estimators':[150] if QUICK_MODE else [300,600,1000]}

def allowed(outcome,strategy,model): return model!='xgboost' or (outcome=='NoHandJointInvolvement' and strategy=='no_explicit_flags')
def estimator_search(model):
    if model=='refined_broad_ridge':
        est=LogisticRegression(l1_ratio=0.0,solver='lbfgs',max_iter=10000,random_state=MASTER_SEED)
        return est,{'model__C':ridge_C},'grid',None
    est=XGBClassifier(objective='binary:logistic',eval_metric='logloss',tree_method='hist',device=XGB_DEVICE,
        random_state=MASTER_SEED,n_jobs=1,scale_pos_weight=1)
    return est,xgb_params,'random',XGB_SEARCH_ITER

def split_plan(data,outcome):
    id_to_idx=pd.Series(np.arange(len(data)),index=data.VisitID).to_dict();rows=[]
    sg=saved_splits.query('outcome==@outcome')
    for repeat in sorted(sg.repeat.unique()):
        rg=sg[sg.repeat.eq(repeat)]
        for fold in sorted(rg.fold.unique()):
            test_ids=rg.loc[rg.fold.eq(fold),'VisitID'].tolist();test=np.array([id_to_idx[x] for x in test_ids],int);test_set=set(test_ids)
            train=np.array([i for i,x in enumerate(data.VisitID) if x not in test_set],int)
            rows.append((int(repeat),int(fold),train,test))
    return rows

def checkpoint(outcome,strategy,model,repeat,fold):
    mode='quick' if QUICK_MODE else 'full'
    return CHECKPOINT_DIR/f'{outcome}__{strategy}__{model}__r{repeat:02d}f{fold:02d}__{mode}.pkl'

In [ ]:
# 6. Fit or resume one outer fold
# Near-complete feature inclusion is learned from the outer-training covariates only.
def fit_fold(outcome,strategy,model,repeat,fold,train_idx,test_idx):
    cp=checkpoint(outcome,strategy,model,repeat,fold)
    if cp.exists() and not FORCE_RECOMPUTE:
        with open(cp,'rb') as f: saved=pickle.load(f)
        if saved.get('run_signature')==RUN_SIGNATURE:
            print('Loading',cp.name);return saved['result']
    data,predictors,semantic,raw=load_outcome(outcome);y=data[outcome].astype(int).to_numpy();ids=data.VisitID.to_numpy()
    selected=list(predictors); explicit_flags=False
    if strategy=='missingness_only':
        X=missingness_matrix(data,predictors);prep='passthrough';selected=list(X.columns)
        training_missing_max=np.nan
    else:
        if strategy=='near_complete':
            rates=data.iloc[train_idx][predictors].isna().mean()
            selected=rates[rates.le(MISSINGNESS_THRESHOLD)].index.tolist()
            training_missing_max=float(rates[selected].max()) if selected else np.nan
            explicit_flags=True
        else:
            training_missing_max=np.nan;explicit_flags=False
        X=data[selected];prep=make_preprocessor(data.iloc[train_idx],selected,semantic,explicit_flags)
    est,params,kind,n_iter=estimator_search(model);pipe=Pipeline([('preprocess',prep),('model',clone(est))],memory=None)
    inner=StratifiedKFold(n_splits=INNER_SPLITS,shuffle=True,random_state=MASTER_SEED+1000*repeat+fold)
    common=dict(estimator=pipe,scoring='neg_log_loss',cv=inner,refit=True,n_jobs=N_JOBS,pre_dispatch='2*n_jobs',error_score='raise',return_train_score=True)
    search=GridSearchCV(param_grid=params,**common) if kind=='grid' else RandomizedSearchCV(param_distributions=params,n_iter=n_iter,
        random_state=MASTER_SEED+1000*repeat+fold,**common)
    t0=time.time()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        with threadpool_limits(limits=1,user_api='blas'): search.fit(X.iloc[train_idx],y[train_idx])
    elapsed=time.time()-t0;conv=[str(w.message) for w in caught if issubclass(w.category,ConvergenceWarning)]
    p=search.best_estimator_.predict_proba(X.iloc[test_idx])[:,1];bi=search.best_index_
    model_name=f'{strategy}__{model}'
    result={'predictions':pd.DataFrame({'VisitID':ids[test_idx],'row_index':test_idx,'outcome':outcome,'strategy':strategy,
        'base_model':model,'model':model_name,'repeat':repeat,'fold':fold,'y':y[test_idx],'probability':p}),
      'diagnostic':{'outcome':outcome,'strategy':strategy,'base_model':model,'model':model_name,'repeat':repeat,'fold':fold,
        'selected_predictor_count':len(selected),'selected_predictors_json':json.dumps(selected,ensure_ascii=False),
        'training_missingness_max':training_missing_max,'explicit_missingness_flags':explicit_flags,
        'convergence_ok':len(conv)==0,'convergence_warning_count':len(conv),'convergence_messages':' | '.join(sorted(set(conv))),
        'all_warning_count':len(caught),'best_inner_log_loss':-float(search.cv_results_['mean_test_score'][bi]),
        'best_inner_train_log_loss':-float(search.cv_results_['mean_train_score'][bi]),
        'best_params_json':json.dumps(search.best_params_,sort_keys=True,default=float),'candidate_count':len(search.cv_results_['params']),
        'fit_seconds':elapsed}}
    with open(cp,'wb') as f: pickle.dump({'run_signature':RUN_SIGNATURE,'result':result},f,pickle.HIGHEST_PROTOCOL)
    print('Saved',cp.name,f'({elapsed:.1f}s)');return result

## Execute the locked repeated nested analysis

In [ ]:
# 7. Execute/resume all permitted sensitivity fits
results=[]
for outcome in OUTCOMES:
    data,_,_,_=load_outcome(outcome)
    for strategy,model in STRATEGIES:
        if not allowed(outcome,strategy,model): continue
        for repeat,fold,train,test in split_plan(data,outcome):
            results.append(fit_fold(outcome,strategy,model,repeat,fold,train,test))
repeated=pd.concat([x['predictions'] for x in results],ignore_index=True)
diagnostics=pd.DataFrame([x['diagnostic'] for x in results])
repeated.to_csv(TABLE_DIR/'missingness_sensitivity_all_repeated_oof_predictions.csv',index=False,encoding='utf-8-sig')
diagnostics.to_csv(TABLE_DIR/'missingness_sensitivity_fit_diagnostics.csv',index=False,encoding='utf-8-sig')
print('Predictions:',repeated.shape,'| fits:',len(diagnostics));display(diagnostics.groupby(['outcome','model']).fit_seconds.agg(['sum','median','max']))

In [ ]:
# 8. Metric, averaging, and paired patient-bootstrap helpers
EPS=1e-8
def calibration_stats(y,p):
    p=np.clip(np.asarray(p,float),EPS,1-EPS);y=np.asarray(y,int)
    if np.std(logit(p))<1e-10:return np.nan,np.nan
    try:
        m=LogisticRegression(C=np.inf,l1_ratio=0.0,solver='lbfgs',max_iter=5000).fit(logit(p).reshape(-1,1),y)
        return float(m.intercept_[0]),float(m.coef_[0,0])
    except Exception:return np.nan,np.nan

def metrics(y,p):
    ci,cs=calibration_stats(y,p)
    return {'auroc':roc_auc_score(y,p),'average_precision':average_precision_score(y,p),'log_loss':log_loss(y,p,labels=[0,1]),
            'brier':brier_score_loss(y,p),'calibration_intercept':ci,'calibration_slope':cs}

def average_repeats(frame):
    keys=['outcome','model','VisitID','y'];extra={}
    if 'strategy' in frame: keys.insert(2,'strategy');keys.insert(3,'base_model')
    return frame.groupby(keys,as_index=False).agg(probability=('probability','mean'),oof_repetitions=('probability','size'),
                                                  probability_sd_across_repeats=('probability','std'))

def paired_difference(a,b,n_boot,seed):
    m=a.merge(b,on=['outcome','VisitID','y'],suffixes=('_a','_b'),validate='one_to_one');y=m.y.to_numpy(int);pa=m.probability_a.to_numpy();pb=m.probability_b.to_numpy()
    pos=np.flatnonzero(y==1);neg=np.flatnonzero(y==0);rng=np.random.default_rng(seed)
    def one(idx):
        ma=metrics(y[idx],pa[idx]);mb=metrics(y[idx],pb[idx])
        return {'auroc':ma['auroc']-mb['auroc'],'average_precision':ma['average_precision']-mb['average_precision'],
                'log_loss':mb['log_loss']-ma['log_loss'],'brier':mb['brier']-ma['brier']}
    est=one(np.arange(len(y)));vals={k:[] for k in est}
    for _ in range(n_boot):
        idx=np.r_[rng.choice(pos,len(pos),True),rng.choice(neg,len(neg),True)];z=one(idx)
        for k,v in z.items():vals[k].append(v)
    return pd.DataFrame([{'metric':k,'improvement_over_reference':v,'ci_low':np.quantile(vals[k],.025),'ci_high':np.quantile(vals[k],.975)} for k,v in est.items()])

In [ ]:
# 9. Patient-averaged estimates, confidence intervals, and paired comparisons
avg=average_repeats(repeated)
orig_avg=average_repeats(original_repeated[original_repeated.model.isin(['refined_broad_ridge','xgboost'])])
perf=[];ci_rows=[];pairs=[]
for (outcome,model),g in avg.groupby(['outcome','model']):
    point=metrics(g.y,g.probability);perf.append({'outcome':outcome,'model':model,'n':len(g),'events':int(g.y.sum()),**point})
    y=g.y.to_numpy(int);p=g.probability.to_numpy();pos=np.flatnonzero(y==1);neg=np.flatnonzero(y==0)
    stable=int(hashlib.sha256(f'{outcome}|{model}'.encode()).hexdigest()[:8],16)%100000;rng=np.random.default_rng(MASTER_SEED+stable);boots=[]
    for _ in range(BOOTSTRAPS):
        idx=np.r_[rng.choice(pos,len(pos),True),rng.choice(neg,len(neg),True)];boots.append(metrics(y[idx],p[idx]))
    boots=pd.DataFrame(boots)
    for metric,value in point.items():
        z=boots[metric].dropna();ci_rows.append({'outcome':outcome,'model':model,'metric':metric,'estimate':value,
            'ci_low':z.quantile(.025),'ci_high':z.quantile(.975),'bootstrap_resamples':BOOTSTRAPS})
    base=g.base_model.iloc[0];ref=orig_avg.query('outcome==@outcome and model==@base')[['outcome','VisitID','y','probability']]
    d=paired_difference(g[['outcome','VisitID','y','probability']],ref,BOOTSTRAPS,MASTER_SEED+stable+500)
    d.insert(0,'reference',base);d.insert(0,'comparator',model);d.insert(0,'outcome',outcome);pairs.append(d)
perf=pd.DataFrame(perf);cis=pd.DataFrame(ci_rows);pairs=pd.concat(pairs,ignore_index=True)
avg.to_csv(TABLE_DIR/'missingness_sensitivity_patient_averaged_predictions.csv',index=False,encoding='utf-8-sig')
perf.to_csv(TABLE_DIR/'missingness_sensitivity_performance.csv',index=False,encoding='utf-8-sig')
cis.to_csv(TABLE_DIR/'missingness_sensitivity_bootstrap_ci.csv',index=False,encoding='utf-8-sig')
pairs.to_csv(TABLE_DIR/'missingness_sensitivity_paired_vs_original.csv',index=False,encoding='utf-8-sig')
display(perf);display(pairs)

## Descriptive missingness-mechanism characterization

In [ ]:
# 10. Predictor-level missingness by outcome status
predictor_rows=[];burden_rows=[];pattern_rows=[];availability=[]
for outcome in OUTCOMES:
    data,predictors,semantic,raw=load_outcome(outcome);y=data[outcome].astype(int)
    miss=data[predictors].isna().astype(int);burden=miss.mean(axis=1)
    for col in predictors:
        p1=float(miss.loc[y.eq(1),col].mean());p0=float(miss.loc[y.eq(0),col].mean());pooled=(p1+p0)/2
        den=np.sqrt(pooled*(1-pooled));smd=(p1-p0)/den if den>0 else 0.0
        predictor_rows.append({'outcome':outcome,'predictor':col,'semantic_type':semantic[col],
          'missing_cases_n':int(miss.loc[y.eq(1),col].sum()),'cases_n':int(y.sum()),'missing_cases_fraction':p1,
          'missing_noncases_n':int(miss.loc[y.eq(0),col].sum()),'noncases_n':int((1-y).sum()),'missing_noncases_fraction':p0,
          'absolute_fraction_difference':abs(p1-p0),'standardized_difference':smd,'overall_missing_fraction':float(miss[col].mean())})
    group=pd.qcut(burden.rank(method='first'),3,labels=['low','middle','high'])
    for level in ['low','middle','high']:
        idx=group.eq(level);burden_rows.append({'outcome':outcome,'missingness_group':level,'n':int(idx.sum()),'events':int(y[idx].sum()),
            'outcome_prevalence':float(y[idx].mean()),'mean_missing_fraction':float(burden[idx].mean()),'min_missing_fraction':float(burden[idx].min()),
            'max_missing_fraction':float(burden[idx].max())})
    pattern=miss.astype(str).agg(''.join,axis=1);tmp=pd.DataFrame({'pattern':pattern,'y':y}).groupby('pattern').y.agg(['size','sum','mean']).reset_index()
    tmp=tmp.sort_values(['size','pattern'],ascending=[False,True]).head(20);tmp.insert(0,'outcome',outcome);tmp.rename(columns={'size':'n','sum':'events','mean':'outcome_prevalence'},inplace=True)
    tmp['pattern_rank']=np.arange(1,len(tmp)+1);pattern_rows.append(tmp)
    true_meta=[c for c in data.columns if any(k in c.lower() for k in ['site','center','centre','calendar','visitdate','visit_date','year']) and c not in ['InitialSiteJoint','InitialSiteSoftTissue']]
    availability.append({'outcome':outcome,'site_or_calendar_fields_found':len(true_meta)>0,'fields':' | '.join(true_meta)})
predictor_missingness=pd.DataFrame(predictor_rows);burden_summary=pd.DataFrame(burden_rows);common_patterns=pd.concat(pattern_rows,ignore_index=True);metadata_availability=pd.DataFrame(availability)
predictor_missingness.to_csv(TABLE_DIR/'predictor_missingness_by_outcome_status.csv',index=False,encoding='utf-8-sig')
burden_summary.to_csv(TABLE_DIR/'outcome_prevalence_by_missingness_burden.csv',index=False,encoding='utf-8-sig')
common_patterns.to_csv(TABLE_DIR/'common_exact_missingness_patterns.csv',index=False,encoding='utf-8-sig')
metadata_availability.to_csv(TABLE_DIR/'site_calendar_metadata_availability.csv',index=False,encoding='utf-8-sig')
display(burden_summary);display(predictor_missingness.sort_values('absolute_fraction_difference',ascending=False).groupby('outcome').head(10));display(metadata_availability)

In [ ]:
# 11. Near-complete predictor selection frequency across outer-training folds
near=diagnostics.query("strategy=='near_complete'").copy();rows=[]
for _,r in near.iterrows():
    selected=set(json.loads(r.selected_predictors_json));data,predictors,_,_=load_outcome(r.outcome)
    for p in predictors: rows.append({'outcome':r.outcome,'repeat':r.repeat,'fold':r.fold,'predictor':p,'selected':p in selected})
near_complete_frequency=pd.DataFrame(rows).groupby(['outcome','predictor'],as_index=False).selected.agg(selection_count='sum',selection_fraction='mean')
near_complete_frequency.to_csv(TABLE_DIR/'near_complete_predictor_selection_frequency.csv',index=False,encoding='utf-8-sig')
display(near_complete_frequency.query('selection_fraction<1').sort_values(['outcome','selection_fraction','predictor']))

## Publication figures

In [ ]:
# 12. Paired performance-change figure
plot=pairs[pairs.metric.isin(['auroc','average_precision','log_loss','brier'])].copy();plot['label']=plot.comparator+' | '+plot.metric
fig,axes=plt.subplots(1,3,figsize=(19,8),sharex=False)
for ax,outcome in zip(axes,OUTCOMES):
    g=plot[plot.outcome.eq(outcome)].reset_index(drop=True);yy=np.arange(len(g))
    ax.hlines(yy,g.ci_low,g.ci_high,color='#2c7fb8',lw=1.8);ax.scatter(g.improvement_over_reference,yy,color='#d95f0e',s=30,zorder=3)
    ax.axvline(0,color='black',ls='--',lw=1);ax.set_yticks(yy,g.label);ax.invert_yaxis();ax.set_title(outcome)
    ax.set_xlabel('Improvement over original locked model\n(positive favors sensitivity model)')
fig.suptitle('Locked missingness sensitivities: paired patient-bootstrap comparisons');fig.tight_layout()
for ext in ['png','svg']:fig.savefig(FIGURE_DIR/f'missingness_sensitivity_paired_change.{ext}',bbox_inches='tight',dpi=400 if ext=='png' else None)
plt.show();plt.close(fig)

In [ ]:
# 13. Missingness-burden prevalence and predictor-association figures
fig,axes=plt.subplots(1,3,figsize=(15,4.8),sharey=False)
for ax,outcome in zip(axes,OUTCOMES):
    g=burden_summary[burden_summary.outcome.eq(outcome)]
    sns.barplot(data=g,x='missingness_group',y='outcome_prevalence',order=['low','middle','high'],color='#2c7fb8',ax=ax)
    for i,r in g.set_index('missingness_group').loc[['low','middle','high']].reset_index().iterrows(): ax.text(i,r.outcome_prevalence+0.01,f"{r.events}/{r.n}",ha='center',fontsize=9)
    ax.set(title=outcome,xlabel='Missingness-burden tertile',ylabel='Outcome prevalence',ylim=(0,max(.55,g.outcome_prevalence.max()+.1)))
fig.suptitle('Outcome prevalence across missingness burden');fig.tight_layout()
for ext in ['png','svg']:fig.savefig(FIGURE_DIR/f'outcome_prevalence_by_missingness_burden.{ext}',bbox_inches='tight',dpi=400 if ext=='png' else None)
plt.show();plt.close(fig)

fig,axes=plt.subplots(1,3,figsize=(17,7))
for ax,outcome in zip(axes,OUTCOMES):
    g=predictor_missingness.query('outcome==@outcome and overall_missing_fraction>0').nlargest(12,'absolute_fraction_difference').sort_values('absolute_fraction_difference')
    ax.barh(g.predictor,g.standardized_difference,color=np.where(g.standardized_difference>=0,'#d95f0e','#2c7fb8'))
    ax.axvline(0,color='black',lw=.8);ax.set(title=outcome,xlabel='Standardized difference\n(positive = more missing among cases)')
fig.suptitle('Largest outcome-associated predictor missingness differences');fig.tight_layout()
for ext in ['png','svg']:fig.savefig(FIGURE_DIR/f'predictor_missingness_outcome_association.{ext}',bbox_inches='tight',dpi=400 if ext=='png' else None)
plt.show();plt.close(fig)

## Automated audit and report export

In [ ]:
# 14. Automated scientific/software audit
checks=[]
def check(cid,description,condition,details=''):checks.append({'check_id':cid,'description':description,'status':'PASS' if bool(condition) else 'FAIL','details':details})
expected_configs=3*3+1;expected_fits=expected_configs*len(ACTIVE_REPEATS)*OUTER_FOLDS
check('M-001','Prior Stage 3B audit fully passed',pd.read_csv(required['stage3b_audit']).status.eq('PASS').all())
check('M-002','Prior Stage 3C audit fully passed',pd.read_csv(required['stage3c_audit']).status.eq('PASS').all())
check('M-003','Exact saved outer split artifact is signed',signature_payload['inputs']['splits']==sha256_file(required['splits']))
check('M-004','Only the three permitted outcomes were analyzed',set(repeated.outcome)==set(OUTCOMES))
check('M-005','OnlyLargeJoints is entirely excluded','OnlyLargeJoints' not in repeated.outcome.unique() and 'OnlyLargeJoints' not in repeated.columns)
check('M-006','All planned outer fits completed',len(diagnostics)==expected_fits,f'{len(diagnostics)}/{expected_fits}')
check('M-007','Every patient has every active repetition',avg.oof_repetitions.eq(len(ACTIVE_REPEATS)).all())
check('M-008','OOF probabilities finite and bounded',np.isfinite(avg.probability).all() and avg.probability.between(0,1).all())
check('M-009','No duplicate patient/repeat predictions',not repeated.duplicated(['outcome','model','VisitID','repeat']).any())
check('M-010','Missingness-only benchmark uses ridge only',set(diagnostics.query("strategy=='missingness_only'").base_model)=={'refined_broad_ridge'})
check('M-011','XGBoost sensitivity is NoHand only',set(diagnostics.query("base_model=='xgboost'").outcome)=={'NoHandJointInvolvement'})
check('M-012','Explicit flags absent in no-flags analyses',not diagnostics.query("strategy=='no_explicit_flags'").explicit_missingness_flags.any())
check('M-013','Near-complete threshold fixed at 5%',MISSINGNESS_THRESHOLD==0.05)
check('M-014','Near-complete inclusion uses training missingness <=5%',diagnostics.query("strategy=='near_complete'").training_missingness_max.le(MISSINGNESS_THRESHOLD+1e-12).all())
check('M-015','All fit diagnostics passed',diagnostics.convergence_ok.all())
check('M-016','Ridge candidate count correct',diagnostics.query("base_model=='refined_broad_ridge'").candidate_count.eq(3 if QUICK_MODE else 21).all())
check('M-017','XGBoost candidate count correct',diagnostics.query("base_model=='xgboost'").candidate_count.eq(XGB_SEARCH_ITER).all())
check('M-018','Natural prevalence retained',True,'No resampling and no class weights in estimator definitions')
check('M-019','Paired comparisons include all sensitivity configurations',pairs.groupby(['outcome','comparator']).ngroups==expected_configs)
check('M-020','Bootstrap count matches mode',BOOTSTRAPS==(100 if QUICK_MODE else 2000),BOOTSTRAPS)
check('M-021','Predictor missingness table covers all outcome predictor sets',len(predictor_missingness)==sum(len(load_outcome(o)[1]) for o in OUTCOMES))
check('M-022','Missingness burden table complete',len(burden_summary)==9)
check('M-023','Site/calendar availability explicitly checked',len(metadata_availability)==3)
check('M-024','No outcome-guided threshold selection',PROTOCOL['analyses']['near_complete_predictors']['threshold']==0.05)
check('M-025','Stage 3B remains primary',PROTOCOL['scientific_hierarchy']['primary'].startswith('Stage 3B'))
check('M-026','Stop rule is frozen','stop internal model development' in PROTOCOL['reporting']['stop_rule'])
check('M-027','All per-fold checkpoints exist',all(checkpoint(o,s,m,r,f).exists() for o in OUTCOMES for s,m in STRATEGIES if allowed(o,s,m) for r in ACTIVE_REPEATS for f in range(OUTER_FOLDS)))
check('M-028','Publication figures saved in PNG and SVG',all((FIGURE_DIR/f'{stem}.{ext}').exists() for stem in ['missingness_sensitivity_paired_change','outcome_prevalence_by_missingness_burden','predictor_missingness_outcome_association'] for ext in ['png','svg']))
audit=pd.DataFrame(checks);audit.to_csv(TABLE_DIR/'stage4_missingness_validation_audit.csv',index=False,encoding='utf-8-sig');display(audit)
if not audit.status.eq('PASS').all():raise AssertionError('Missingness-addendum audit failed')
print(f'ALL {len(audit)} MISSINGNESS-ADDENDUM CHECKS PASSED')

In [ ]:
# 15. Artifact registry and run summary
artifacts=[]
for folder in [TABLE_DIR,FIGURE_DIR,CHECKPOINT_DIR,OUT_DIR]:
    for p in sorted(folder.glob('*')):
        if p.is_file() and p.name!='artifact_registry.csv':artifacts.append({'filename':p.name,'path':str(p),'size_bytes':p.stat().st_size,'sha256':sha256_file(p)})
registry=pd.DataFrame(artifacts).drop_duplicates('path');registry.to_csv(TABLE_DIR/'artifact_registry.csv',index=False,encoding='utf-8-sig')
summary={'run_signature':RUN_SIGNATURE,'mode':'quick' if QUICK_MODE else 'full','active_repeats':ACTIVE_REPEATS,'outer_folds':OUTER_FOLDS,
 'inner_folds':INNER_SPLITS,'bootstraps':BOOTSTRAPS,'missingness_threshold':MISSINGNESS_THRESHOLD,'all_audits_passed':True,
 'scientific_status':'post hoc locked missingness-mechanism addendum; Stage 3B unchanged and primary',
 'stop_rule':'No further internal model development after this addendum.'}
with open(OUT_DIR/'stage4_missingness_run_summary.json','w',encoding='utf-8') as f:json.dump(summary,f,indent=2)
display(registry[['filename','size_bytes']])

## Interpretation rules

- High missingness-only performance means that local testing/documentation patterns carry substantial outcome information.
- Deterioration after removing explicit numeric/binary indicators indicates dependence on those workflow signals. Categorical missing values retain the frozen `Unknown`-category handling and must be described transparently.
- The near-complete analysis is a robustness sensitivity, not a newly optimized model. Predictor eligibility is learned from outer-training covariates only.
- Stable performance in these sensitivities increases confidence but does not establish external transportability.
- No sensitivity result may replace or reinterpret Stage 3B as the prespecified primary analysis.
- After this analysis, stop internal model development and proceed to manuscript preparation and future external validation.

In [ ]:
# 16. FINAL CELL — checkpoint-backed self-contained HTML report
if os.environ.get('RA_STAGE4M_EXPORT_MODE')=='1':
    print('HTML subprocess mode: recursive export skipped.')
else:
    candidates=[PROJECT_DIR/NOTEBOOK_NAME,ENV_PROJECT/NOTEBOOK_NAME,COLAB_PROJECT/NOTEBOOK_NAME,Path.cwd()/NOTEBOOK_NAME,
                Path.cwd()/'ra_analysis'/NOTEBOOK_NAME,Path.cwd()/NOTEBOOK_NAME]
    notebook_path=next((p.resolve() for p in candidates if p.exists()),None)
    if notebook_path is None:raise FileNotFoundError(f'Save notebook as {NOTEBOOK_NAME} before export')
    env=os.environ.copy();env['RA_STAGE4M_EXPORT_MODE']='1';env.pop('RA_STAGE4M_FORCE_RECOMPUTE',None)
    report=REPORT_DIR/'Stage4_missingness_addendum_report.html'
    cmd=[sys.executable,'-m','jupyter','nbconvert','--to','html','--execute','--ExecutePreprocessor.timeout=0',
         '--HTMLExporter.embed_images=True','--output',report.name,'--output-dir',str(REPORT_DIR),str(notebook_path)]
    print('Executing checkpoint-backed report export...')
    result=subprocess.run(cmd,env=env,cwd=str(notebook_path.parent),text=True,capture_output=True)
    if result.stdout:print(result.stdout)
    if result.returncode!=0:print(result.stderr);raise RuntimeError(f'HTML export failed: {result.returncode}')
    if not report.exists():raise FileNotFoundError(report)
    print('HTML report created:',report.resolve())